In [1]:
from dotenv import load_dotenv
from gitsource import GithubRepositoryDataReader
from minsearch import Index
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()


In [2]:
# Collect the data

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [3]:
#1: How many lesson pages are there? 
len(files)

72

In [4]:
files[0]

RawRepositoryFile(filename='01-agentic-rag/lessons/01-intro.md', content='# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are 

In [5]:
#Process the data easy indexing
processed_data = [{"filename": file.filename, "content": file.content} for file in files]

processed_data[0]

{'filename': '01-agentic-rag/lessons/01-intro.md',
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is th

In [6]:
def build_index(documents):
    index = Index(
        text_fields=["content"], 
        keyword_fields=["filename"]
    )
    index.fit(documents)
    return index

In [7]:
index = build_index(documents=processed_data)

In [8]:
#Question 2: What's the filename of the first result of the query below? 
question = "How does the agentic loop keep calling the model until it stops?"
index.search(query=question, num_results=1)

[{'filename': '01-agentic-rag/lessons/14-agentic-loop.md',
  'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent

In [9]:
from rag_helper import RAGBase

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the content from the filenames.
Use only the facts from the CONTEXT when answering the QUESTION.
If you don't know, say, "I don't know." 
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [10]:
question = "How does the agentic loop keep calling the model until it stops?"

output = assistant.rag(query=question)


In [11]:
# Question 3: How many input tokens did we send for the request above? 
print(output['input_tokens'])

7115


In [12]:
from gitsource import chunk_documents

chunks = chunk_documents(processed_data, size=2000, step=1000)

In [13]:
#Question 4: How many chunks did we generate? 
len(chunks)

295

In [14]:
#Question 5: Rag with Chunking

index_chunked = build_index(documents=chunks)

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the content from the filenames.
Use only the facts from the CONTEXT when answering the QUESTION.
If you don't know, say, "I don't know." 
""".strip()

assistant = RAGBase(
    index=index_chunked,
    llm_client=openai_client,
    instructions=instructions,
)

In [15]:
question = "How does the agentic loop keep calling the model until it stops?"

output = assistant.rag(query=question)

In [23]:
print(output['input_tokens'])

print(f"{round(2298/7115, 3)} is approximately 1/3 fewer.")

2298
0.323 is approximately 1/3 fewer.


In [24]:
#Question 6
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [25]:
def search(query:str) -> dict[str,str]:
    """
    Search the chunked course lessons for the answers to the users queries.

    Args:
        query (str): Query from the user

    Returns:
        dict[str,str]: _description_
    """

    results = index_chunked.search(
            query, 
            num_results=5,
        )

    return results
    

In [26]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [27]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the chunked course lessons for the answers to the users queries.\n\nArgs:\n    query (str): Query from the user\n\nReturns:\n    dict[str,str]: _description_',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [28]:
instructions = "You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."

In [29]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [30]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received
